# Taxi Trip Data Cleaning

Steps:
1. Load raw CSV
2. Type conversions & trip end filling
3. Rename to snake_case
4. Impute lat/lon from census tract
5. Impute census tract from lat/lon (lookup)
6. Drop rows with no spatial data
7. Payment type cleanup
8. Company standardization
9. Duplicate check
10. Remove invalid trips (any zero/below-threshold)
11. Consistency & logic checks
12. Absolute outlier filtering
13. Save cleaned parquet
14. ID remapping + compact parquet
15. Time features from trip_start
16. Cyclic encoding (sin/cos)
17. Electronic payment fee detection + save compact parquet
18. Save removed rows

## 0) Setup & Configuration

In [2]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

PROJECT = Path(os.getcwd()).parent
DATA_DIR = PROJECT / "data"

CSV_PATH = DATA_DIR / "Taxi_Trips_(2024-)_20260512.csv"
OUTPUT_CLEAN = DATA_DIR / "Taxi_Trips_cleaned.parquet"
OUTPUT_COMPACT = DATA_DIR / "Taxi_Trips_compact.parquet"
OUTPUT_TRIP_ID_MAP = DATA_DIR / "trip_id_mapping.csv"
OUTPUT_TAXI_ID_MAP = DATA_DIR / "taxi_id_mapping.csv"
OUTPUT_REMOVED = DATA_DIR / "Removed_Rows.parquet"
OUTPUT_NO_SPATIAL = DATA_DIR / "No_Spatial_Value.parquet"
OUTPUT_COMPANIES = DATA_DIR / "company_names_after_cleaning.csv"

removed_frames = []

def flag_and_remove(df, mask, reason):
    n = mask.sum()
    if n == 0:
        return df
    removed = df.loc[mask].copy()
    removed["removal_reason"] = reason
    removed_frames.append(removed)
    print(f"  Removed {n:,} rows: {reason}")
    return df[~mask].reset_index(drop=True)

print("Setup complete.")
print(f"Project: {PROJECT}")
print(f"Data dir: {DATA_DIR}")

Setup complete.
Project: l:\My Drive\Business Analytics_drive\Tripple A\Tripple A
Data dir: l:\My Drive\Business Analytics_drive\Tripple A\Tripple A\data


---
## 1) LOAD

In [3]:
df = pd.read_csv(CSV_PATH, low_memory=False)
original_rows = df.shape[0]
print(f"Loaded {original_rows:,} rows, {df.shape[1]} columns")
print(df.dtypes)

Loaded 15,406,960 rows, 23 columns
Trip ID                           str
Taxi ID                           str
Trip Start Timestamp              str
Trip End Timestamp                str
Trip Seconds                      str
Trip Miles                        str
Pickup Census Tract           float64
Dropoff Census Tract          float64
Pickup Community Area         float64
Dropoff Community Area        float64
Fare                              str
Tips                              str
Tolls                             str
Extras                            str
Trip Total                        str
Payment Type                      str
Company                           str
Pickup Centroid Latitude      float64
Pickup Centroid Longitude     float64
Pickup Centroid Location          str
Dropoff Centroid Latitude     float64
Dropoff Centroid Longitude    float64
Dropoff Centroid  Location        str
dtype: object


---
## 2) TYPE CONVERSIONS + TRIP END FILLING

In [4]:
money_cols = ["Fare", "Tips", "Tolls", "Extras", "Trip Total"]
for col in money_cols:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
    )

df["Trip Seconds"] = pd.to_numeric(
    df["Trip Seconds"].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)
df["Trip Seconds"] = df["Trip Seconds"].round().astype("Int64")

df["Trip Miles"] = pd.to_numeric(
    df["Trip Miles"].astype(str).str.replace(r"[^0-9.\-]", "", regex=True), errors="coerce"
)

df["Trip Start Timestamp"] = pd.to_datetime(df["Trip Start Timestamp"], errors="coerce")
df["Trip End Timestamp"] = pd.to_datetime(df["Trip End Timestamp"], errors="coerce")

mask_fill_end = (
    df["Trip End Timestamp"].isna()
    & df["Trip Start Timestamp"].notna()
    & df["Trip Seconds"].notna()
)
df.loc[mask_fill_end, "Trip End Timestamp"] = (
    df.loc[mask_fill_end, "Trip Start Timestamp"]
    + pd.to_timedelta(df.loc[mask_fill_end, "Trip Seconds"], unit="s")
)

print(f"Filled {mask_fill_end.sum():,} missing Trip End Timestamps")
print(f"Rows: {len(df):,}")

C:\Users\behna\AppData\Local\Temp\ipykernel_85520\3958006670.py:16: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Trip Start Timestamp"] = pd.to_datetime(df["Trip Start Timestamp"], errors="coerce")
C:\Users\behna\AppData\Local\Temp\ipykernel_85520\3958006670.py:17: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Trip End Timestamp"] = pd.to_datetime(df["Trip End Timestamp"], errors="coerce")


Filled 0 missing Trip End Timestamps
Rows: 15,406,960


---
## 3) RENAME TO SNAKE_CASE

In [5]:
snake_map = {
    "Trip ID": "trip_id",
    "Taxi ID": "taxi_id",
    "Trip Start Timestamp": "trip_start",
    "Trip End Timestamp": "trip_end",
    "Trip Seconds": "trip_seconds",
    "Trip Miles": "trip_miles",
    "Pickup Census Tract": "pickup_census_tract",
    "Dropoff Census Tract": "dropoff_census_tract",
    "Pickup Community Area": "pickup_community_area",
    "Dropoff Community Area": "dropoff_community_area",
    "Fare": "fare_usd",
    "Tips": "tips_usd",
    "Tolls": "tolls_usd",
    "Extras": "extras_usd",
    "Trip Total": "trip_total_usd",
    "Payment Type": "payment_type",
    "Company": "company",
    "Pickup Centroid Latitude": "pickup_lat",
    "Pickup Centroid Longitude": "pickup_lon",
    "Pickup Centroid Location": "pickup_location",
    "Dropoff Centroid Latitude": "dropoff_lat",
    "Dropoff Centroid Longitude": "dropoff_lon",
    "Dropoff Centroid  Location": "dropoff_location",
}
df.rename(columns=snake_map, inplace=True)
print(f"Renamed {len(snake_map)} columns.")
print(df.columns.tolist())

Renamed 23 columns.
['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare_usd', 'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type', 'company', 'pickup_lat', 'pickup_lon', 'pickup_location', 'dropoff_lat', 'dropoff_lon', 'dropoff_location']


---
## 4) LAT/LON FILL VIA CENSUS TRACT
Use known census tract centroids (median lat/lon) to fill missing lat/lon.

In [6]:
df["pickup_census_tract"] = df["pickup_census_tract"].astype("Int64").astype(str).replace("<NA>", np.nan)
df["dropoff_census_tract"] = df["dropoff_census_tract"].astype("Int64").astype(str).replace("<NA>", np.nan)

pu_tract_map = (
    df[df["pickup_census_tract"].notna() & df["pickup_lat"].notna()]
    .groupby("pickup_census_tract")[["pickup_lat", "pickup_lon"]]
    .median()
)
do_tract_map = (
    df[df["dropoff_census_tract"].notna() & df["dropoff_lat"].notna()]
    .groupby("dropoff_census_tract")[["dropoff_lat", "dropoff_lon"]]
    .median()
)

mask_pu = df["pickup_lat"].isna() & df["pickup_census_tract"].notna()
df.loc[mask_pu, "pickup_lat"] = df.loc[mask_pu, "pickup_census_tract"].map(pu_tract_map["pickup_lat"])
df.loc[mask_pu, "pickup_lon"] = df.loc[mask_pu, "pickup_census_tract"].map(pu_tract_map["pickup_lon"])

mask_do = df["dropoff_lat"].isna() & df["dropoff_census_tract"].notna()
df.loc[mask_do, "dropoff_lat"] = df.loc[mask_do, "dropoff_census_tract"].map(do_tract_map["dropoff_lat"])
df.loc[mask_do, "dropoff_lon"] = df.loc[mask_do, "dropoff_census_tract"].map(do_tract_map["dropoff_lon"])

print(f"Filled {mask_pu.sum():,} pickup lat/lon from census tract")
print(f"Filled {mask_do.sum():,} dropoff lat/lon from census tract")

Filled 2,023 pickup lat/lon from census tract
Filled 16,403 dropoff lat/lon from census tract


---
## 5) CENSUS TRACT IMPUTATION VIA LOOKUP (lat/lon -> census tract)
Since lat/lon are centroid coordinates of census tracts, identical (lat, lon) pairs
always map to the same census tract. We build a simple lookup dictionary from rows
where both lat/lon and census tract are known, then map missing tracts.

In [7]:
for prefix in ["pickup", "dropoff"]:
    lat_col = f"{prefix}_lat"
    lon_col = f"{prefix}_lon"
    tract_col = f"{prefix}_census_tract"

    known = df[df[tract_col].notna() & df[lat_col].notna() & df[lon_col].notna()]
    lookup = known.groupby([lat_col, lon_col])[tract_col].first().to_dict()
    print(f"{prefix}: Built lookup with {len(lookup):,} unique (lat,lon) -> census tract pairs")

    missing = df[df[tract_col].isna() & df[lat_col].notna() & df[lon_col].notna()]
    if len(missing) == 0:
        print(f"{prefix}: No missing census tracts to impute.")
        continue

    keys = list(zip(missing[lat_col], missing[lon_col]))
    mapped = pd.Series([lookup.get(k, np.nan) for k in keys], index=missing.index)
    filled = mapped.notna().sum()
    df.loc[missing.index, tract_col] = mapped

    print(f"{prefix}: Imputed {filled:,} / {len(missing):,} missing census tracts via lookup")

pickup: Built lookup with 610 unique (lat,lon) -> census tract pairs
pickup: Imputed 5,381 / 8,128,006 missing census tracts via lookup
dropoff: Built lookup with 657 unique (lat,lon) -> census tract pairs
dropoff: Imputed 5,928 / 7,482,644 missing census tracts via lookup


---
## 6) DROP ROWS WITH NO SPATIAL DATA
Remove rows where ALL spatial columns are null. Save to separate parquet.

In [8]:
spatial_cols = [
    "pickup_census_tract", "dropoff_census_tract",
    "pickup_community_area", "dropoff_community_area",
    "pickup_lat", "pickup_lon", "pickup_location",
    "dropoff_lat", "dropoff_lon", "dropoff_location",
]

no_spatial_mask = df[spatial_cols].isna().all(axis=1)
no_spatial_count = no_spatial_mask.sum()
df[no_spatial_mask].to_parquet(str(OUTPUT_NO_SPATIAL), index=False)
df = df[~no_spatial_mask].reset_index(drop=True)

print(f"Removed {no_spatial_count:,} rows with no spatial data at all")
print(f"Remaining: {len(df):,} rows")

Removed 279,212 rows with no spatial data at all
Remaining: 15,127,748 rows


---
## 7) PAYMENT TYPE CLEANUP

In [9]:
df["payment_type"] = (
    df["payment_type"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
)
print(f"Payment types: {df['payment_type'].nunique()}")
print(df["payment_type"].value_counts())

Payment types: 8
payment_type
credit card    5553409
cash           3794159
mobile         3554075
prcard         1650607
unknown         526895
no charge        40623
dispute           7916
prepaid             64
Name: count, dtype: int64


---
## 8) COMPANY STANDARDIZATION

In [10]:
df["company"] = (
    df["company"].str.lower().str.strip().str.replace(r"\s+", " ", regex=True)
)
df["company"] = df["company"].str.replace(r"[,\.]", "", regex=True)
df["company"] = df["company"].str.strip().str.replace(r"\s+", " ", regex=True)

manual_mappings = {
    "choice taxi association inc": "choice taxi association",
    "blue ribbon taxi association inc": "blue ribbon taxi association",
    "top cab affiliation": "top cab",
    "medallion leasin": "medallion leasing",
}
df["company"] = df["company"].replace(manual_mappings)

company_export = df["company"].value_counts().reset_index()
company_export.columns = ["company_name", "count"]
company_export.to_csv(str(OUTPUT_COMPANIES), index=False)
print(f"Unique companies: {df['company'].nunique()}")

Unique companies: 39


---
## 9) DUPLICATE CHECK (trip_id)

In [11]:
total = len(df)
unique = df["trip_id"].nunique()
dups = total - unique
print(f"Total rows: {total:,}, Unique trip_ids: {unique:,}, Duplicates: {dups:,}")

if dups > 0:
    dup_mask = df.duplicated(subset="trip_id", keep="first")
    df = flag_and_remove(df, dup_mask, "duplicate_trip_id")
else:
    print("No duplicates found.")

Total rows: 15,127,748, Unique trip_ids: 15,127,748, Duplicates: 0
No duplicates found.


---
## 10) REMOVE INVALID TRIPS
Remove if **any** of:
- `trip_seconds < 60` (under 1 minute)
- `trip_miles < 0.01` (under 0.01 miles)
- `fare_usd == 0` (zero fare)

All removed rows go into `Removed_Rows.parquet` with a reason.

In [12]:
neg_cols = ["trip_seconds", "trip_miles", "fare_usd", "tips_usd", "tolls_usd", "extras_usd", "trip_total_usd"]
for col in neg_cols:
    df = flag_and_remove(df, df[col] < 0, f"negative_{col}")

df = flag_and_remove(
    df,
    (df["trip_seconds"].notna()) & (df["trip_seconds"] < 60),
    "trip_duration_under_1min",
)

df = flag_and_remove(
    df,
    (df["trip_miles"].notna()) & (df["trip_miles"] < 0.01),
    "trip_distance_under_0.01mi",
)

df = flag_and_remove(
    df,
    (df["fare_usd"].notna()) & (df["fare_usd"] == 0),
    "fare_zero",
)

print(f"\nRemaining after invalid trip removal: {len(df):,}")

  Removed 770,833 rows: trip_duration_under_1min
  Removed 608,189 rows: trip_distance_under_0.01mi
  Removed 4,638 rows: fare_zero

Remaining after invalid trip removal: 13,744,088


---
## 11) CONSISTENCY & LOGIC CHECKS

In [13]:
df = flag_and_remove(
    df,
    df["trip_end"] < df["trip_start"],
    "time_travel_end_before_start",
)

ts_safe = df["trip_seconds"].fillna(0).to_numpy(dtype="float64")
tm_safe = df["trip_miles"].fillna(0).to_numpy(dtype="float64")
speed_mph = np.where(ts_safe > 0, tm_safe / (ts_safe / 3600), 0)
df = flag_and_remove(
    df,
    speed_mph > 100,
    "impossible_speed_over_100mph",
)

print(f"Remaining after consistency checks: {len(df):,}")

  Removed 76 rows: time_travel_end_before_start


C:\Users\behna\AppData\Local\Temp\ipykernel_85520\4063605070.py:9: RuntimeWarning: divide by zero encountered in divide
  speed_mph = np.where(ts_safe > 0, tm_safe / (ts_safe / 3600), 0)
C:\Users\behna\AppData\Local\Temp\ipykernel_85520\4063605070.py:9: RuntimeWarning: invalid value encountered in divide
  speed_mph = np.where(ts_safe > 0, tm_safe / (ts_safe / 3600), 0)


  Removed 1,845 rows: impossible_speed_over_100mph
Remaining after consistency checks: 13,742,167


---
## 12) ABSOLUTE OUTLIER FILTERING
Hard cutoffs:
- `trip_miles > 200` -> remove
- `trip_seconds > 7200` (> 2h) -> remove
- `fare_usd < 3.25` (below basefare) -> remove
- No upper cut on fare

In [14]:
df = flag_and_remove(
    df,
    (df["trip_miles"].notna()) & (df["trip_miles"] > 200),
    "trip_miles_over_200",
)

df = flag_and_remove(
    df,
    (df["trip_seconds"].notna()) & (df["trip_seconds"] > 7200),
    "trip_seconds_over_2h",
)

df = flag_and_remove(
    df,
    (df["fare_usd"].notna()) & (df["fare_usd"] < 3.25),
    "fare_below_basefare_3.25",
)

print(f"\nRemaining after absolute outlier filtering: {len(df):,}")

  Removed 172 rows: trip_miles_over_200
  Removed 25,255 rows: trip_seconds_over_2h
  Removed 2,012 rows: fare_below_basefare_3.25

Remaining after absolute outlier filtering: 13,714,728


---
## 13) DROP LOCATION TEXT COLUMNS & SAVE CLEANED PARQUET
Keep census tract columns (needed later). Drop only the point-location text columns.

In [15]:
cols_to_drop = ["pickup_location", "dropoff_location"]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f"Final columns: {df.columns.tolist()}")
print(f"Final rows: {len(df):,}")

df.to_parquet(str(OUTPUT_CLEAN), index=False)
print(f"Saved {OUTPUT_CLEAN}")

Final columns: ['trip_id', 'taxi_id', 'trip_start', 'trip_end', 'trip_seconds', 'trip_miles', 'pickup_census_tract', 'dropoff_census_tract', 'pickup_community_area', 'dropoff_community_area', 'fare_usd', 'tips_usd', 'tolls_usd', 'extras_usd', 'trip_total_usd', 'payment_type', 'company', 'pickup_lat', 'pickup_lon', 'dropoff_lat', 'dropoff_lon']
Final rows: 13,714,728
Saved l:\My Drive\Business Analytics_drive\Tripple A\Tripple A\data\Taxi_Trips_cleaned.parquet


---
## 14) ID REMAPPING + COMPACT PARQUET

In [16]:
trip_id_map = pd.DataFrame({
    "trip_id_int": range(1, len(df) + 1),
    "trip_id_orig": df["trip_id"].values,
})

taxi_ids_unique = df["taxi_id"].dropna().unique()
taxi_id_map = pd.DataFrame({
    "taxi_id_int": range(1, len(taxi_ids_unique) + 1),
    "taxi_id_orig": taxi_ids_unique,
})
taxi_lookup = dict(zip(taxi_id_map["taxi_id_orig"], taxi_id_map["taxi_id_int"]))

df["trip_id_int"] = range(1, len(df) + 1)
df["taxi_id_int"] = df["taxi_id"].map(taxi_lookup)

df.drop(columns=["trip_id", "taxi_id"], inplace=True)

cols_order = ["trip_id_int", "taxi_id_int"] + [c for c in df.columns if c not in ("trip_id_int", "taxi_id_int")]
df = df[cols_order]

trip_id_map.to_csv(str(OUTPUT_TRIP_ID_MAP), index=False)
taxi_id_map.to_csv(str(OUTPUT_TAXI_ID_MAP), index=False)

print(f"trip_id_map: {len(trip_id_map):,} rows")
print(f"taxi_id_map: {len(taxi_id_map):,} rows")

trip_id_map: 13,714,728 rows
taxi_id_map: 3,315 rows


---
## 15) TIME FEATURES FROM trip_start

In [17]:
ts = df["trip_start"]

df["hour"] = ts.dt.hour.astype("int8")
df["day_of_week"] = ts.dt.dayofweek.astype("int8")
df["month"] = ts.dt.month.astype("int8")
df["date"] = ts.dt.date
df["is_weekend"] = df["day_of_week"].isin([5, 6])

df["bin_30min"] = ts.dt.floor("30min")
df["bin_1h"] = ts.dt.floor("1h")
df["bin_4h"] = ts.dt.floor("4h")
df["bin_1d"] = ts.dt.normalize()
df["bin_1w"] = ts.dt.to_period("W").apply(lambda p: p.start_time)

time_cols = ["hour", "day_of_week", "month", "date", "is_weekend",
            "bin_30min", "bin_1h", "bin_4h", "bin_1d", "bin_1w"]

print(f"Added {len(time_cols)} time columns.")
df[time_cols].sample(10, random_state=42)

Added 10 time columns.


,hour,day_of_week,month,date,is_weekend,bin_30min,bin_1h,bin_4h,bin_1d,bin_1w
782593,7,2,3,2026-03-18,False,2026-03-18 07:30:00,2026-03-18 07:00:00,2026-03-18 04:00:00,2026-03-18,2026-03-16
3038347,9,3,10,2025-10-23,False,2025-10-23 09:00:00,2025-10-23 09:00:00,2025-10-23 08:00:00,2025-10-23,2025-10-20
1413212,19,3,2,2026-02-05,False,2026-02-05 19:30:00,2026-02-05 19:00:00,2026-02-05 16:00:00,2026-02-05,2026-02-02
4165670,4,4,8,2025-08-22,False,2025-08-22 04:30:00,2025-08-22 04:00:00,2025-08-22 04:00:00,2025-08-22,2025-08-18
10348336,14,4,8,2024-08-09,False,2024-08-09 14:00:00,2024-08-09 14:00:00,2024-08-09 12:00:00,2024-08-09,2024-08-05
2117531,11,1,12,2025-12-16,False,2025-12-16 11:00:00,2025-12-16 11:00:00,2025-12-16 08:00:00,2025-12-16,2025-12-15
13080135,16,2,2,2024-02-21,False,2024-02-21 16:30:00,2024-02-21 16:00:00,2024-02-21 16:00:00,2024-02-21,2024-02-19
12969461,9,3,2,2024-02-29,False,2024-02-29 09:00:00,2024-02-29 09:00:00,2024-02-29 08:00:00,2024-02-29,2024-02-26
6854457,18,1,3,2025-03-25,False,2025-03-25 18:00:00,2025-03-25 18:00:00,2025-03-25 16:00:00,2025-03-25,2025-03-24
4393014,13,5,8,2025-08-09,True,2025-08-09 13:30:00,2025-08-09 13:00:00,2025-08-09 12:00:00,2025-08-09,2025-08-04


---
## 16) CYCLIC ENCODING (sin/cos)

In [18]:
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
df["dow_sin"] = np.sin(2 * np.pi * df["day_of_week"] / 7)
df["dow_cos"] = np.cos(2 * np.pi * df["day_of_week"] / 7)
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

cyclic_cols = ["hour", "hour_sin", "hour_cos",
              "day_of_week", "dow_sin", "dow_cos",
              "month", "month_sin", "month_cos"]

print(f"Added 6 cyclic encoding columns.")
df[cyclic_cols].sample(10, random_state=42)

Added 6 cyclic encoding columns.


,hour,hour_sin,hour_cos,day_of_week,dow_sin,dow_cos,month,month_sin,month_cos
782593,7,0.965926,-2.588190e-01,2,0.974928,-0.222521,3,1.000000e+00,6.123234e-17
3038347,9,0.707107,-7.071068e-01,3,0.433884,-0.900969,10,-8.660254e-01,5.000000e-01
1413212,19,-0.965926,2.588190e-01,3,0.433884,-0.900969,2,8.660254e-01,5.000000e-01
4165670,4,0.866025,5.000000e-01,4,-0.433884,-0.900969,8,-8.660254e-01,-5.000000e-01
10348336,14,-0.500000,-8.660254e-01,4,-0.433884,-0.900969,8,-8.660254e-01,-5.000000e-01
2117531,11,0.258819,-9.659258e-01,1,0.781831,0.623490,12,-2.449294e-16,1.000000e+00
13080135,16,-0.866025,-5.000000e-01,2,0.974928,-0.222521,2,8.660254e-01,5.000000e-01
12969461,9,0.707107,-7.071068e-01,3,0.433884,-0.900969,2,8.660254e-01,5.000000e-01
6854457,18,-1.000000,-1.836970e-16,1,0.781831,0.623490,3,1.000000e+00,6.123234e-17
4393014,13,-0.258819,-9.659258e-01,5,-0.974928,-0.222521,8,-8.660254e-01,-5.000000e-01


---
## 17) ELECTRONIC PAYMENT FEE DETECTION + SAVE COMPACT PARQUET

In [19]:
df["has_electronic_fee"] = (
    df["payment_type"].isin(["credit card", "mobile"])
    & ((df["trip_total_usd"] - df[["fare_usd","tips_usd","tolls_usd","extras_usd"]].sum(axis=1)).round(2) > 0.49)
    & ((df["trip_total_usd"] - df[["fare_usd","tips_usd","tolls_usd","extras_usd"]].sum(axis=1)).round(2) < 0.51)
)

df.to_parquet(str(OUTPUT_COMPACT), index=False)
print(f"Saved compact parquet: {OUTPUT_COMPACT}")
print(f"has_electronic_fee: {df['has_electronic_fee'].sum():,} rows")

Saved compact parquet: l:\My Drive\Business Analytics_drive\Tripple A\Tripple A\data\Taxi_Trips_compact.parquet
has_electronic_fee: 6,425,634 rows


---
## 18) SAVE REMOVED ROWS

In [20]:
if removed_frames:
    removed_df = pd.concat(removed_frames, ignore_index=True)
    removed_df.to_parquet(str(OUTPUT_REMOVED), index=False)
    print(f"Saved {len(removed_df):,} removed rows to {OUTPUT_REMOVED}")
    print(f"\nRemoval breakdown:")
    print(removed_df["removal_reason"].value_counts().to_string())
else:
    print("No rows were removed.")

print(f"\n=== CLEANING SUMMARY ===")
print(f"Original rows:  {original_rows:,}")
print(f"Final rows:     {len(df):,}")
removed_total = original_rows - len(df)
print(f"Removed:        {removed_total:,} ({removed_total/original_rows*100:.2f}%)")

Saved 1,413,020 removed rows to l:\My Drive\Business Analytics_drive\Tripple A\Tripple A\data\Removed_Rows.parquet

Removal breakdown:
removal_reason
trip_duration_under_1min        770833
trip_distance_under_0.01mi      608189
trip_seconds_over_2h             25255
fare_zero                         4638
fare_below_basefare_3.25          2012
impossible_speed_over_100mph      1845
trip_miles_over_200                172
time_travel_end_before_start        76

=== CLEANING SUMMARY ===
Original rows:  15,406,960
Final rows:     13,714,728
Removed:        1,692,232 (10.98%)
